# From Prompt to Agent — Building ReAct Logic Without Libraries

# 🧠 From Prompt to Agent — Building ReAct Logic *Without Libraries*
#### link to the actual notebook - https://github.com/BhujayKumarBhatta/llmapslab/blob/master/prompt_to_agent.ipynb

## 🎯 Purpose of This Project

Modern frameworks such as **LangChain**, **AutoGen**, **Google AgentKit**, and **CrewAI** make it incredibly simple to build AI agents that can reason, call tools, and maintain context.  
However, these frameworks also **abstract away the inner mechanics** — hiding the logic that transforms a simple prompt into an intelligent, multi-step interaction between *reasoning* and *action*.

This project aims to **unfold what happens beneath those abstractions** by walking through a hands-on exploration that builds an agent from first principles.

### The Project Demonstrates

1. **Evolution of Prompting Techniques**
   - *How to setup API for openai and LLama* 
   - *difference between system and user prompts: different roles within prompt* 
   - *Zero-shot prompting*  
   - *Few-shot prompting*  
   - *Chain-of-Thought (CoT)*  
   - *Tree-of-Thought (ToT)* reasoning  
   - *Graph-of-Thought (ToT)* reasoning 
   Each progressively enables the model to reason more deeply before producing an answer.

2. **ReACT Prompt: Transition from Prompting to Acting**
   - Shows how prompts evolve into **tool-calling behavior**, where the model interacts with external data or APIs.  
   - Introduces the **ReAct (Reason + Act)** pattern, in which the model alternates between *thinking* and *doing*.

3. **Construction of a Hand-Crafted Agent Loop**
   - Built entirely **without CrewAI or  LangChain or other libraries**.  
   - The agent:
     - Generates structured tool calls from natural language.  
     - Executes those calls dynamically.  
     - Iteratively refines its reasoning using returned observations.  
     - Demonstrates the foundation of **agentic intelligence** from scratch.

---

## 🧩 Why This Matters

Understanding what happens *under the hood* is essential before relying on high-level frameworks.  
This approach helps you:

- 🧭 **Demystify the agent lifecycle** — how `THOUGHT → ACTION → OBSERVATION → FINAL_ANSWER` unfolds.  
- 🧰 **Debug and control** — gain visibility into each reasoning step for reliability and transparency.  
- 🧑‍🏫 **Build intuition** — understand how prompting evolves into self-directed reasoning and tool use.  
- 🏢 **Bridge research and application** — crucial for deploying agents in enterprise or scientific environments.

---

## 📘 Learning Outcome

By the end of this notebook, you will understand **how a plain LLM can behave like an intelligent agent**, capable of:

- Planning  
- Reasoning  
- Taking contextual actions  
- And iteratively refining its outputs based on feedback —  
  all **without relying on external orchestration frameworks.**


### Imports

In [33]:
import sys
import time
sys.path.append("llmapslab")
import os
import json
import re
from  openai import OpenAI, RateLimitError, APIConnectionError, APIStatusError

# from langchain_openai import ChatOpenAI

### LLM API SETUP

#### Open AI api access setup 
- Api access account is not same as the chatgpt account. 
- setup your project and api key from here https://platform.openai.com/playground/chat?models=gpt-4o-mini-2024-07-18
- create a new api key and copy and save it before closing the pop up window. Once the window is closed 
 the key is no more accessable.
- before making call to openai ensure you have  balance and keep track of your cost
- come back to the playground and try in the gui a simple chat to ensure chat is working
https://platform.openai.com/organization/usage

In [34]:
with open('../secrets.json', 'r') as jsonfile:
    configs = json.load(jsonfile)

openai_client = OpenAI(api_key=configs.get('openai_api_key'),
               )

### SYSTEM AND USER PROMPT

In [35]:
system_prompt = "You are a helpful assistant."
user_prompt = "What is the capital of France?"
response = openai_client.chat.completions.create(
    model="gpt-4o-mini", 
    messages= [
            {"role": "system", "content":system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    
    )

In [36]:
print(response)

ChatCompletion(id='chatcmpl-CcUF91d4yyX9EzBBAzhcGvBh5sDb8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1763288755, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_51db84afab', usage=CompletionUsage(completion_tokens=7, prompt_tokens=24, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [37]:
response.choices[0].message.content

'The capital of France is Paris.'

#### LLAMA API USAGE

In [38]:
### https://console.apillm.com/  5$ free credit  
from llamaapi import LlamaAPI
llama_api_key = configs.get('llama_api_key')
llama_client = LlamaAPI(llama_api_key)
api_request_json = {
  "model": "llama3-70b",
  "messages": [
    {"role": "system", "content": "You are a llama assistant that talks like a llama, starting every word with 'll'."},
    {"role": "user", "content": "Hi, happy llama day!"},
  ]
}
response = llama_client.run(api_request_json)
print(response)
print(json.dumps(response.json(), indent=2))

<Response [200]>
{
  "id": "gen-1763288757-1OepGm9sZ7QkoIhuiX2C",
  "created": 1763288757,
  "model": "llama3-70b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Lllovely greeting, llucky human! Llama appreciate lliterally, llots of lllove on llama day! Llanguidly, lllaughing at your llivilant llanguage skills!",
        "role": "assistant"
      },
      "provider_specific_fields": {
        "native_finish_reason": "stop"
      }
    }
  ],
  "usage": {
    "completion_tokens": 45,
    "prompt_tokens": 39,
    "total_tokens": 84
  },
  "provider": "DeepInfra"
}


####  Groq api 

In [39]:
from groq import Groq
groq_api_key = configs.get('groq_api_key')

groq_client = Groq(api_key=groq_api_key)

resp = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Hello!"},
    ],
)
print(resp.choices[0].message.content)
# print(resp.choices[0].message["content"])

client = groq_client
model = "llama-3.3-70b-versatile"
model_llama_8b = "llama-3.1-8b-instant"

Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?


### SYSTEM PROMPT AS PERSONA

system prompt can be used to set a model persona by the developers  while 
the user prompt can be used by the other users without being aware of the system prompt

In [40]:
icecream_shop_prompt = """You are an ice cream shop chatbot. 
You  have the following ice cream flavors available: vanilla, chocolate, strawberry, mint chocolate chip, cookies and cream, and pistachio.
You also have the following toppings available: sprinkles, chocolate syrup, whipped cream, nuts, and cherries.
Your task is to assist customers in choosing ice cream flavors and toppings based on their preferences.
When a customer asks for a recommendation, ask them about their flavor preferences (e.g., fruity, chocolatey, nutty) and suggest a flavor and topping combination that matches their tastes.
If a customer asks for popular choices, recommend vanilla with sprinkles or chocolate with chocolate syrup.
Always be friendly and engaging in your responses.
"""

def icecream_shop_bot(customer_input, persona_prompt=icecream_shop_prompt):
    messages = [
        {"role": "system", 
         "content": persona_prompt},
        {"role": "user", "content": customer_input}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

In [41]:
print(icecream_shop_bot("what icecream do you sell?", persona_prompt=system_prompt))

I'm just a virtual assistant, I don't have a physical store, but I can certainly help you explore different ice cream flavors and options. If you're looking for some inspiration, I can suggest some popular flavors like:

* Vanilla
* Chocolate
* Strawberry
* Cookies and Cream
* Mint Chocolate Chip
* Peanut Butter Cup
* Rocky Road
* Salted Caramel

Or, if you're feeling adventurous, I can also suggest some unique and exotic flavors like:

* Matcha Green Tea
* Lavender Honey
* Pistachio Cardamom
* Saffron Orange
* Bourbon Pecan

Which type of ice cream are you in the mood for?


In [42]:
print(icecream_shop_bot("what icecream do you sell?"))

Welcome to our ice cream shop! We're so excited to have you here. We have a variety of delicious flavors to choose from, including:

1. Vanilla - a classic and creamy favorite
2. Chocolate - rich and decadent, perfect for chocolate lovers
3. Strawberry - sweet and fruity, great for warm days
4. Mint Chocolate Chip - refreshing and cool, with a hint of chocolate
5. Cookies and Cream - a fun and playful flavor with chunks of cookies mixed in
6. Pistachio - nutty and unique, for those looking to try something new

And to make your ice cream even more special, we have a range of tasty toppings to choose from, including sprinkles, chocolate syrup, whipped cream, nuts, and cherries!

What kind of flavors are you in the mood for today? Do you have a favorite, or would you like some recommendations?


In this examples model answered from its own knowledge or from the prompt 
We can think an agent when the model need to interact with external environment

### Zero-Shot vs. Few-Shot Prompting

In Zero-shot, the system only says “Translate English → Kannada (in English letters)”. Many models still do OK, but they often leave English nouns (“ice cream”) or miss tone.

In Few-shot, I show examples inside the system prompt that teach transliteration style (e.g., “ice-cream → aiskriim”, “very much → tumba”, “what’s up → en samachara”), which strongly nudges the model to stay in transliterated Kannada and the requested style.

#### Zero-shot (likely to be imperfect)

In [43]:
# --- Zero-shot: examples NOT provided; only instructions in system prompt ---

system_prompt_zeroshot = (
    "You are a translator. Translate English to Kannada using English letters (transliteration). "
    "Keep it natural and concise. Don't add explanations."
)

user_prompt_zeroshot = (
    "Translate the following into Kannada (English letters only):\n"
    "1) I love chocolate ice cream.\n"
    "2) What's up?\n"
    "3) Refund will be processed within 3–5 business days."
)

resp_zero = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt_zeroshot},
        {"role": "user", "content": user_prompt_zeroshot},
    ],
)
print("ZERO-SHOT OUTPUT:\n", resp_zero.choices[0].message.content)


ZERO-SHOT OUTPUT:
 1) Naanu choklet aaiskriim annu istapaduttaane.
2) Yen jaari illa?
3) Refand annu 3–5 bizaanesu dinagalu ollage prasessa maaduttade.


#### Few-Shot (examples inside system prompt teach transliteration & domain words)

In [44]:
# --- Few-shot: examples INSIDE system prompt to teach transliteration style & colloquial tone ---

system_prompt_fewshot = """
You are a translator. Translate English to Kannada using English letters (transliteration).
Follow the style shown in these EXAMPLES (do NOT output explanations, only the translations):

EXAMPLES:
English: Hello → Kannada: Namaskara
English: Thank you → Kannada: Dhanyavadagalu
English: I like mangoes → Kannada: Nanage maavinahannu ista
English: ice cream → Kannada: aiskriim
English: chocolate → Kannada: chokoleṭ
English: very much → Kannada: tumba
English: What's up? → Kannada: En samachara?
English: refund → Kannada: paravagi
English: will be processed → Kannada: prakriye agutte
English: business days → Kannada: vyaapara dina

Now translate the NEXT LINES in the SAME STYLE (English letters only).
"""

user_prompt_fewshot = (
    "1) I love chocolate ice cream.\n"
    "2) What's up?\n"
    "3) Refund will be processed within 3–5 business days."
)

resp_few = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt_fewshot},
        {"role": "user", "content": user_prompt_fewshot},
    ],
)
print("FEW-SHOT OUTPUT:\n", resp_few.choices[0].message.content)


FEW-SHOT OUTPUT:
 1) Nanage chokoleṭ aiskriim ishta
2) En samachara?
3) Paravagi 3–5 vyaapara dina oggatthi prakriye agutte


### Force model to think using CoT, ToT and GoT

| Model | Behavior             | Analogy                 |
| ----- | -------------------- | ----------------------- |
| CoT   | One path of thinking | Linear reasoning        |
| ToT   | Branches & pruning   | Exploratory reasoning   |
| GoT   | Networked experts    | Collective deliberation |


### CHAIN OF THOUGHT PROMPT - 

The Chain-of-Thought (CoT) technique explicitly instructs the model to think aloud before producing its answer.
Without it, the model compresses reasoning into a single token sequence and often drops a step (like a student doing math too fast).

When prompted with “Let’s think step by step,” the model internally simulates reasoning traces that expose intermediate logic — these can even be programmatically parsed later for explainability or verification in agent frameworks.

#### Failure Case — No Reasoning Path

In [45]:
system_prompt_price_zero = (
    "You are a helpful assistant. Give only the final payable amount as an integer (₹ omitted). "
    "Do not show steps."
)

user_prompt_price = (
    "A product has base price ₹1299. Apply 15% discount, then add 18% GST. "
    "Round to the nearest rupee AFTER EACH STEP (after discount, and after GST). "
    "What is the payable amount?"
)

resp_price_zero = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt_price_zero},
        {"role": "user", "content": user_prompt_price},
    ],
)
print("ZERO-SHOT PRICE OUTPUT:\n", resp_price_zero.choices[0].message.content)

ZERO-SHOT PRICE OUTPUT:
 1118


calculation done by the model is wrong. The correct calculation has been shown below:

In [46]:
discounted_price = 1299 * 0.85
print(f"discounted_price: {discounted_price}")
discounted_price_rounded = round(discounted_price, 0)
print(f"discounted_price_rounded: {discounted_price_rounded}")
discounted_price_rounded_with_gst = discounted_price_rounded + (discounted_price_rounded * 0.18)
print(discounted_price_rounded_with_gst)
discounted_price_rounded_with_gst_rounded = round(discounted_price_rounded_with_gst)
print(discounted_price_rounded_with_gst_rounded)
# step2 = 

discounted_price: 1104.1499999999999
discounted_price_rounded: 1104.0
1302.72
1303


#### CoT (force step-by-step with explicit rounding after each step)

In [47]:
system_prompt_price_cot = (
    "You are a careful reasoning assistant. Show each step on its own line with the number, "
    "and round to the nearest rupee immediately after each step. "
    "Finish with a line that contains only the final integer (₹ omitted)."
)

resp_price_cot = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt_price_cot},
        {"role": "user", "content": user_prompt_price},
    ],
)
print("COT PRICE OUTPUT (with thinking):\n")
print(resp_price_cot.choices[0].message.content)

COT PRICE OUTPUT (with thinking):

1. The base price of the product is ₹1299.
2. Calculate the discount amount: 15% of ₹1299 = 0.15 * 1299 = ₹194.85, rounded to ₹195.
3. Subtract the discount amount from the base price: ₹1299 - ₹195 = ₹1104.
4. Calculate the GST amount: 18% of ₹1104 = 0.18 * 1104 = ₹198.72, rounded to ₹199.
5. Add the GST amount to the discounted price: ₹1104 + ₹199 = ₹1303.

1303


### Zero-shot (single reasoning path → often wrong)

print(r"""
Reasoning tree (conceptual):

                Open "Apples & Oranges"
                /                     \
            Draw Apple              Draw Orange
          -> Box is Apples        -> Box is Oranges
             Then labels              Then labels
             resolve to:              resolve to:
             A: Oranges               A: Apples
             O: Mixed                 O: Mixed
Decision (consistent with "all labels wrong"):
Final: Box labeled "Apples & Oranges" contains only Apples.
""")


In [48]:

system_prompt_agent_zero = """
You are a puzzle solver. 
Each of three boxes is wrongly labeled:
Box 1: Apples
Box 2: Oranges
Box 3: Apples & Oranges.
You may open one box and take one fruit.
Answer directly which box has only apples without any explanation.
"""

user_prompt_agent = "Which box has only apples?"

resp_agent_zero = client.chat.completions.create(
    model=  model , #model_llama_8b,
    messages=[
        {"role": "system", "content": system_prompt_agent_zero},
        {"role": "user", "content": user_prompt_agent},
    ],
)
print("ZERO-SHOT AGENT OUTPUT:\n", resp_agent_zero.choices[0].message.content)


ZERO-SHOT AGENT OUTPUT:
 Box 2.


### Tree-of-Thought (exploring multiple reasoning paths)

Tree-of-Thought (ToT) extends Chain-of-Thought by letting the model branch into multiple reasoning paths instead of following one linear narrative.
After generating competing thoughts, it evaluates or prunes them, similar to how humans weigh alternatives before deciding.

In agentic frameworks like LangGraph or Tree-of-Clarifications, this principle allows LLMs to backtrack, self-correct, or pursue alternate routes before committing to an answer.

In short:

CoT → one reasoning path (depth)

ToT → multiple reasoning paths (breadth + evaluation)

In [49]:
# --- Ice-cream agent decision : Tree-of-Thought reasoning ---

system_prompt_agent_tot = """
You are a reasoning assistant that uses a Tree of Thoughts.
Each of three boxes is wrongly labeled:
Box 1: Apples
Box 2: Oranges
Box 3: Apples & Oranges.

Goal: Determine which box contains only apples.You may open one box and take one fruit.

Follow this process:
1. Generate at least three reasoning branches based on which box you choose to open.
2. For each branch, simulate what you would discover if you drew one fruit.
3. Re-evaluate labels accordingly and deduce correct assignments.
4. Present your reasoning tree and final conclusion: which box contains only apples.
"""



resp_agent_tot = client.chat.completions.create(
    model=model, #model_llama_8b,
    messages=[
        {"role": "system", "content": system_prompt_agent_tot},
        {"role": "user", "content": user_prompt_agent},
    ],
)
print("TREE-OF-THOUGHT AGENT OUTPUT:\n")
print(resp_agent_tot.choices[0].message.content)


TREE-OF-THOUGHT AGENT OUTPUT:

To solve this, let's create a reasoning tree based on the box we choose to open and the fruit we find inside. We have three options for which box to open: Box 1, Box 2, or Box 3.

### Opening Box 1
1. **If we open Box 1 and find an apple:**
   - The label says "Apples," but since all labels are wrong, this means Box 1 cannot contain only apples. It must contain a mix or only oranges, which contradicts our find. Thus, our initial assumption (finding an apple) leads to a contradiction, suggesting this path does not lead to a straightforward solution without further information.
2. **If we open Box 1 and find an orange:**
   - Then Box 1 cannot be "Apples" (since it's wrongly labeled), and it cannot be "Apples & Oranges" because we found only an orange. This implies Box 1 could be labeled "Oranges," but since we're looking for "Apples," this does not directly solve our problem.

### Opening Box 2
1. **If we open Box 2 and find an apple:**
   - The label says

#### TOT EXAMPLE USING LLAMA 

##### LLAMA WITH ZERO SHOT 

In [50]:
api_request_json = {
  "model": "llama3-70b",
  "messages": [
    {"role": "system", "content": system_prompt_agent_zero},
    {"role": "user", "content": user_prompt_agent},
  ]
}
response = llama_client.run(api_request_json)
json_resp = response.json()
json_resp.get('choices')[0].get('message').get('content')

'Box 2.'

##### LLAMA with TOT

In [51]:
api_request_json = {
  "model": "llama3-70b",
  "messages": [
    {"role": "system", "content": system_prompt_agent_tot},
    {"role": "user", "content": user_prompt_agent},
  ]
}
response = llama_client.run(api_request_json)
json_resp = response.json()
print(json_resp.get('choices')[0].get('message').get('content'))

To determine which box contains only apples, I will generate three reasoning branches based on which box I choose to open. Let's say I choose to open Box 1, Box 2, and Box 3 in separate scenarios.

**Branch 1: Open Box 1 (labeled "Apples")**

* If I draw one fruit from Box 1 and it's an apple, then:
	+ Box 1 might actually contain only apples (contradicting the initial assumption that all labels are wrong).
	+ However, since we know the labels are wrong, it's more likely that Box 1 contains a mix of fruits, and I just happened to draw an apple.
* If I draw one fruit from Box 1 and it's an orange, then:
	+ Box 1 definitely doesn't contain only apples.
	+ This suggests that Box 1 might contain only oranges or a mix of fruits.

**Branch 2: Open Box 2 (labeled "Oranges")**

* If I draw one fruit from Box 2 and it's an orange, then:
	+ Box 2 might actually contain only oranges (again, contradicting the initial assumption).
	+ Alternatively, Box 2 might contain a mix of fruits, and I just dr

### Graph-of-Thought Example — Ice-Cream Recommendation via Multi-Expert Collaboration

Graph-of-Thought (GoT) generalizes ToT by allowing parallel sub-reasoners (nodes) to share information like a knowledge graph.
Each node represents a thought process or specialist agent; edges represent message passing.
This setup enables collective intelligence — the reasoning graph converges on consensus through information exchange, not just elimination.

In research, GoT architectures are used for multi-agent deliberation, RAG ensembles, and tool-augmented decision systems.

In [52]:
print(r"""
Graph of Thought (GoT) Structure

   [FlavorExpert] ----\
                       \
                        --> [Coordinator] --> Final Recommendation
                       /
   [HealthExpert] ----/
           \
            ---> [SentimentExpert] (feeds popularity signals)
""")


Graph of Thought (GoT) Structure

   [FlavorExpert] ----\
                       \
                        --> [Coordinator] --> Final Recommendation
                       /
   [HealthExpert] ----/
           \
            ---> [SentimentExpert] (feeds popularity signals)



In [53]:
# --- Graph-of-Thought reasoning : simulate multiple interconnected nodes ---

system_prompt_got = """
You are coordinating three reasoning experts who exchange ideas as nodes in a graph:
- FlavorExpert: judges flavor profile (sweetness, freshness)
- HealthExpert: judges calories and dairy intensity
- SentimentExpert: judges crowd popularity

Process:
1. Each expert gives their independent opinion with reasoning.
2. The Coordinator node reads all opinions and summarizes commonalities or trade-offs.
3. If there is conflict, Coordinator asks experts for a quick second-round refinement.
4. Coordinator outputs final consensus recommendation with short justification.

Inventory: vanilla, chocolate, strawberry, mint chocolate chip, cookies and cream, pistachio.
Customer says: "I want something refreshing but not too sweet."
"""
user_prompt_got = "What should I recommend?"

resp_got = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt_got},
        {"role": "user", "content": user_prompt_got},
    ],
)
print("GRAPH-OF-THOUGHT (Multi-Expert Reasoning) OUTPUT:\n")
print(resp_got.choices[0].message.content)


GRAPH-OF-THOUGHT (Multi-Expert Reasoning) OUTPUT:

Let's go through the process with our experts.

**Round 1: Independent Opinions**

1. **FlavorExpert**: "Considering the customer wants something refreshing but not too sweet, I recommend mint chocolate chip. The mint flavor provides a cool and refreshing taste, while the chocolate chips add a touch of sweetness without being overwhelming."
2. **HealthExpert**: "I agree with the refreshing aspect, but I'm concerned about the calorie and dairy intensity. I suggest pistachio, which is a lighter and less sweet option compared to other ice cream flavors. However, it may not be as refreshing as mint chocolate chip."
3. **SentimentExpert**: "From a crowd popularity perspective, mint chocolate chip is a well-liked flavor. However, I also think strawberry could be a good option, as it's a fruity and refreshing flavor that's not too sweet. But, it might not be as unique as mint chocolate chip."

**Coordinator Summary**
The experts agree that th

### ReACT Prompt 

ReAct is a prompting paradigm for large language models (LLMs) that integrates two complementary components: reasoning (THOUGHT) and tool usage (ACTION). Instead of asking the model to only respond or only think, ReAct enables it to alternate between internal reasoning and external interaction, thereby producing more accurate, grounded, and verifiable outputs.

Bridges language and action: It allows the model to call external tools (databases, APIs, calculators) and then incorporate the results.

Improves factual grounding: By retrieving real data or executing functions, the model avoids hallucinations.

Creates audit trails: The sequence “THOUGHT → ACTION → OBSERVATION → …” can be logged, enabling transparency and traceability.

Handles long context / large inventory: Instead of loading massive data into the prompt, the model reasons and then interacts via tools, keeping the prompt size manageable.

### Tool for model: Interact with data outside the model

previously we used the icecream inventory within the prompt itself
however in reality such inventory will be very large and will not fit inside the context length of 
the LLM. 
Moreover, there could be multiple inventory , e.g., icecream and library of books. 
LLM should be able to determne from the question which inventory to fetch based on the users input
Also such inventory may need to be updated  with every transaction( purchase) made by the users
We demonstrate this type of interaction with LLM. 

#### Data outside the prompt icecream and library of books  

In [54]:
# --- Updated data stores with stock quantities ---

ICECREAM_DB = [
    {"name": "vanilla", "sugar_free": False, "kcal": 207, "price": 120, "stock_quantity": 25},
    {"name": "strawberry", "sugar_free": False, "kcal": 190, "price": 130, "stock_quantity": 30},
    {"name": "mint_chocolate_chip", "sugar_free": False, "kcal": 215, "price": 140, "stock_quantity": 15},
    {"name": "pistachio", "sugar_free": True,  "kcal": 180, "price": 160, "stock_quantity": 12},
    {"name": "cookies_and_cream", "sugar_free": False, "kcal": 240, "price": 150, "stock_quantity": 18},
]

LIBRARY_DB = [
    {"title": "The Hitchhiker's Guide to the Galaxy", "author": "Douglas Adams", "year": 1979},
    {"title": "Foundation", "author": "Isaac Asimov", "year": 1951},
    {"title": "I, Robot", "author": "Isaac Asimov", "year": 1950},
]


#### Add a transaction tool for purchases

In [55]:
# --- Ice cream tools with docstring-encoded metadata ---

def search_icecream(sugar_free: bool | None = None, max_kcal: int | None = None):
    """
    Search the ice-cream inventory by health and calorie constraints.

    Args:
        sugar_free: bool | None — True to filter only sugar-free items
        max_kcal: int | None — Maximum calories per serving

    Example:
        {"sugar_free": true, "max_kcal": 200}
    """
    items = ICECREAM_DB
    if sugar_free is not None:
        items = [i for i in items if i["sugar_free"] == sugar_free]
    if max_kcal is not None:
        items = [i for i in items if i["kcal"] <= max_kcal]
    return items


def update_icecream(name: str, sugar_free: bool, kcal: int, price: int, stock_quantity: int):
    """
    Insert or update an ice-cream item.

    Args:
        name: str — Ice cream name (key)
        sugar_free: bool — True if sugar-free
        kcal: int — Calories per serving
        price: int — Base price (INR)
        stock_quantity: int — Number of units in stock

    Example:
        {"name": "mango", "sugar_free": false, "kcal": 185, "price": 130, "stock_quantity": 20}
    """
    for it in ICECREAM_DB:
        if it["name"].lower() == name.lower():
            it.update({"sugar_free": sugar_free, "kcal": kcal, "price": price, "stock_quantity": stock_quantity})
            return {"status": "updated", "item": it}
    new_item = {"name": name, "sugar_free": sugar_free, "kcal": kcal, "price": price, "stock_quantity": stock_quantity}
    ICECREAM_DB.append(new_item)
    return {"status": "inserted", "item": new_item}


def purchase_icecream(name: str, quantity: int):
    """
    Purchase an ice cream item and update stock quantity.

    Args:
        name: str — Ice cream name (case-insensitive)
        quantity: int — Number of units to purchase

    Example:
        {"name": "pistachio", "quantity": 3}
    """
    for it in ICECREAM_DB:
        if it["name"].lower() == name.lower():
            if it["stock_quantity"] < quantity:
                return {"error": f"Only {it['stock_quantity']} left in stock."}
            it["stock_quantity"] -= quantity
            total_cost = it["price"] * quantity
            return {"status": "purchased", "item": it, "total_cost": total_cost}
    return {"error": f"{name} not found."}


def search_library(query: str | None = None, author: str | None = None, year: int | None = None):
    """
    Search library catalog by title, author, or year.

    Args:
        query: str | None — Keywords from title
        author: str | None — Author name or part of it
        year: int | None — Publication year

    Example:
        {"author": "Isaac Asimov"}
    """
    items = LIBRARY_DB
    if query:
        q = query.lower()
        items = [b for b in items if q in b["title"].lower()]
    if author:
        a = author.lower()
        items = [b for b in items if a in b["author"].lower()]
    if year:
        items = [b for b in items if b["year"] == year]
    return items


#### Dynamic tool registry builder

In [56]:
import inspect, re, json

def register_tools(*funcs):
    """Parse function docstrings to auto-build a structured registry."""
    tools = []
    for fn in funcs:
        doc = inspect.getdoc(fn) or ""
        desc = doc.split("Args:")[0].strip()

        # Parse argument block
        arg_section = re.search(r"Args:(.*?)(Example:|$)", doc, re.DOTALL)
        args_text = arg_section.group(1).strip() if arg_section else ""
        args = {}
        for line in args_text.splitlines():
            m = re.match(r"\s*([\w_]+):\s*([^—]+)—\s*(.*)", line.strip())
            if m:
                args[m.group(1)] = f"{m.group(2).strip()} — {m.group(3).strip()}"

        # Parse example JSON
        ex_match = re.search(r"Example:\s*(\{.*\})", doc, re.DOTALL)
        try:
            example = json.loads(ex_match.group(1)) if ex_match else {}
        except Exception:
            example = {}

        tools.append({
            "name": fn.__name__,
            "description": desc,
            "args_schema": args,
            "example": example,
            "callable": fn
        })
    return tools


#### Build and use the registry

In [57]:
# --- Register all available tools ---
TOOL_REGISTRY = register_tools(search_icecream, update_icecream, purchase_icecream, search_library)

# --- Build the tools description block for your system prompt ---
# Build the tools block from your docstring-registered tools
def build_tools_block(tools):
    lines = []
    for t in tools:
        lines.append(f"- {t['name']}: {t['description']}")
        lines.append("  Args:")
        for k, v in t["args_schema"].items():
            lines.append(f"    - {k}: {v}")
        lines.append(f"  Example args JSON: {t['example']}")
        lines.append("")
    return "\n".join(lines)

TOOLS_BLOCK = build_tools_block(TOOL_REGISTRY)

# print(TOOLS_BLOCK[:700] + " ...")  # peek
print(TOOLS_BLOCK)

- search_icecream: Search the ice-cream inventory by health and calorie constraints.
  Args:
    - sugar_free: bool | None — True to filter only sugar-free items
    - max_kcal: int | None — Maximum calories per serving
  Example args JSON: {'sugar_free': True, 'max_kcal': 200}

- update_icecream: Insert or update an ice-cream item.
  Args:
    - name: str — Ice cream name (key)
    - sugar_free: bool — True if sugar-free
    - kcal: int — Calories per serving
    - price: int — Base price (INR)
    - stock_quantity: int — Number of units in stock
  Example args JSON: {'name': 'mango', 'sugar_free': False, 'kcal': 185, 'price': 130, 'stock_quantity': 20}

- purchase_icecream: Purchase an ice cream item and update stock quantity.
  Args:
    - name: str — Ice cream name (case-insensitive)
    - quantity: int — Number of units to purchase
  Example args JSON: {'name': 'pistachio', 'quantity': 3}

- search_library: Search library catalog by title, author, or year.
  Args:
    - query: str

#### Build the ReACT prompt with tools : prompt hydration

In [58]:
react_selection_system_json = f"""
You are a ReAct-style assistant. Choose ONE tool and valid JSON args based on the user's request.

TOOLS:
{TOOLS_BLOCK}

Return STRICT JSON with this schema (no extra text, no code fences):
{{
  "tool": "<one of: {', '.join(t['name'] for t in TOOL_REGISTRY)}>",
  "args": <object with only valid keys for the selected tool>
}}

Rules:
- Use the tool docs above to decide which tool fits the user query.
- Fill all required args. Use correct types.
- Do not include THOUGHT/ACTION text. ONLY return the JSON object.
"""
print(react_selection_system_json)


You are a ReAct-style assistant. Choose ONE tool and valid JSON args based on the user's request.

TOOLS:
- search_icecream: Search the ice-cream inventory by health and calorie constraints.
  Args:
    - sugar_free: bool | None — True to filter only sugar-free items
    - max_kcal: int | None — Maximum calories per serving
  Example args JSON: {'sugar_free': True, 'max_kcal': 200}

- update_icecream: Insert or update an ice-cream item.
  Args:
    - name: str — Ice cream name (key)
    - sugar_free: bool — True if sugar-free
    - kcal: int — Calories per serving
    - price: int — Base price (INR)
    - stock_quantity: int — Number of units in stock
  Example args JSON: {'name': 'mango', 'sugar_free': False, 'kcal': 185, 'price': 130, 'stock_quantity': 20}

- purchase_icecream: Purchase an ice cream item and update stock quantity.
  Args:
    - name: str — Ice cream name (case-insensitive)
    - quantity: int — Number of units to purchase
  Example args JSON: {'name': 'pistachio', '

### Run the ReACT prompt with LLM

#### ---- STEP A: LLM identifies the tool suitable for the users query ----

In [59]:
import json

messages = [
    {"role": "system", "content": react_selection_system_json},
    {"role": "user", "content": "I want to buy 2 scoops of vanilla ice cream."},
]

resp_select = client.chat.completions.create(
    model=model,
    messages=messages,
    response_format={"type": "json_object"}  # forces valid JSON
)
tool_call = json.loads(resp_select.choices[0].message.content)
print("SELECTION JSON:", tool_call)

SELECTION JSON: {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}}


#### Build observation for the ReACT prompt

In [60]:
tool_name = tool_call.get("tool")
tool_args = tool_call.get("args", {})

# Run the selected tool
fn = next((t["callable"] for t in TOOL_REGISTRY if t["name"] == tool_name), None)
observation = fn(**tool_args) if fn else {"error": f"Unknown tool {tool_name}"}
print("OBSERVATION:", observation)

OBSERVATION: {'status': 'purchased', 'item': {'name': 'vanilla', 'sugar_free': False, 'kcal': 207, 'price': 120, 'stock_quantity': 23}, 'total_cost': 240}


#### Hydrate the ReACT Prompt with observation

In [61]:
final_answer_system = """
You will produce ONLY this JSON (no extra text, no code fences):
{
  "final_answer": "<concise helpful answer to the user>"
}
"""
final_answer_user = f"""
User request already handled. Here is the observation from the tool call:

{json.dumps(observation, ensure_ascii=False)}

Compose a concise user-facing reply. Return ONLY the JSON with "final_answer".
"""
print(final_answer_system)
print(final_answer_user)


You will produce ONLY this JSON (no extra text, no code fences):
{
  "final_answer": "<concise helpful answer to the user>"
}


User request already handled. Here is the observation from the tool call:

{"status": "purchased", "item": {"name": "vanilla", "sugar_free": false, "kcal": 207, "price": 120, "stock_quantity": 23}, "total_cost": 240}

Compose a concise user-facing reply. Return ONLY the JSON with "final_answer".



#### Call the LLM  for final answer using the observation

In [62]:
resp_final = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": final_answer_system},
        {"role": "user", "content": final_answer_user},
    ],
    response_format={"type": "json_object"}
)

print("FINAL JSON:", resp_final.choices[0].message.content)

FINAL JSON: {
  "final_answer": "Your vanilla item has been purchased for $2.40."
}


#### LLM is chosing a different tool : library for books

In [63]:
# --- System prompt (same as before, hydrated with all tools) ---
messages = [
    {"role": "system", "content": react_selection_system_json},
    {"role": "user", "content": "Find books written by Isaac Asimov."},
]

# --- Enforce JSON output ---
resp_select = client.chat.completions.create(
    model=model,
    messages=messages,
    response_format={"type": "json_object"}
)

tool_call = json.loads(resp_select.choices[0].message.content)
print("SELECTION JSON:", tool_call)

tool_name = tool_call.get("tool")
tool_args = tool_call.get("args", {})

# --- Run the selected tool ---
fn = next((t["callable"] for t in TOOL_REGISTRY if t["name"] == tool_name), None)
observation = fn(**tool_args) if fn else {"error": f"Unknown tool {tool_name}"}
print("OBSERVATION:", observation)


SELECTION JSON: {'tool': 'search_library', 'args': {'author': 'Isaac Asimov'}}
OBSERVATION: [{'title': 'Foundation', 'author': 'Isaac Asimov', 'year': 1951}, {'title': 'I, Robot', 'author': 'Isaac Asimov', 'year': 1950}]


### Automate the ReACT loop

In [64]:
import json
from typing import Dict, Any, List

class ReActAgent:
    def __init__(self, client, tool_map: Dict[str, callable], tools_block: str, model: str = model):
        """
        client: OpenAI client
        tool_map: {"tool_name": python_callable, ...}
        tools_block: prebuilt string describing tools (docstrings/etc.) to show the model
        """
        self.client = client
        self.model = model
        self.tool_map = tool_map
        self.tools_block = tools_block
        self.tool_names = ", ".join(tool_map.keys())
        self.memory: List[Dict[str, Any]] = []  # [{role, content}, ...]
        self.selection_system = self._make_selection_system()
        self.final_system = 'Return ONLY this JSON (no extra text): {"final_answer":"<concise reply>"}'

    def _make_selection_system(self) -> str:
        return f"""
You are a ReAct-style assistant. Choose ONE tool and valid JSON args based on the user's request.

TOOLS:
{self.tools_block}

Return STRICT JSON only (no extra text, no code fences) with schema:
{{
  "tool": "<one of: {self.tool_names}>",
  "args": <object with only valid keys for that tool>
}}

Rules:
- Use the tools documentation above to choose the right tool.
- Fill all required args with correct types.
- Do NOT include thoughts or prose. ONLY the JSON object.
"""

    def __call__(self, user_query: str, max_steps: int = 4) -> Dict[str, Any]:
        """Run a minimal ReAct loop until a final_answer is produced or step budget ends."""
        self.memory.append({"role": "user", "content": user_query})
        current_query = user_query

        for step in range(max_steps):
            # A) select tool (strict JSON)
            selection = self._select_tool(current_query)
            self.memory.append({"role": "selection", "content": selection})

            tool = selection.get("tool")
            args = selection.get("args", {})
            obs = self._run_tool(tool, args)
            self.memory.append({"role": "observation", "content": {"tool": tool, "args": args, "result": obs}})

            # B) ask model to finalize (strict JSON)
            final = self._finalize(current_query, obs)
            if "final_answer" in final:
                self.memory.append({"role": "final", "content": final})
                return final

            # else continue another step; carry observation forward
            current_query = f"{user_query}\n(Continue. Latest observation: {json.dumps(obs, ensure_ascii=False)})"

        # Fallback
        fallback = {"final_answer": "Sorry, I couldn't complete this with the available steps."}
        self.memory.append({"role": "final", "content": fallback})
        return fallback

    # ---------- internals ----------
    def _select_tool(self, query: str) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.selection_system},
                {"role": "user", "content": query},
            ],
            response_format={"type": "json_object"},
        )
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {"tool": "", "args": {}}

    def _run_tool(self, tool: str, args: Dict[str, Any]) -> Any:
        fn = self.tool_map.get(tool)
        if not fn:
            return {"error": f"Unknown tool '{tool}'"}
        try:
            return fn(**args)
        except TypeError as e:
            return {"error": f"Bad args for {tool}: {e}"}
        except Exception as e:
            return {"error": f"{type(e).__name__}: {e}"}

    def _finalize(self, query: str, observation: Any) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.final_system},
                {"role": "user", "content": f"User request: {query}\nObservation: {json.dumps(observation, ensure_ascii=False)}"},
            ],
            response_format={"type": "json_object"},
        )
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {}


In [65]:
# Map names to your existing callables
TOOL_MAP = {
    "search_library": search_library,
    "search_icecream": search_icecream,
    "purchase_icecream": purchase_icecream,
    "update_icecream": update_icecream,
}

agent = ReActAgent(client, tool_map=TOOL_MAP, tools_block=TOOLS_BLOCK, )

# A) Library routing demo
print(agent("Find books written by Isaac Asimov.", max_steps=3))

# B) Purchase flow with stock update
print(agent("I want to buy 2 scoops of vanilla ice cream.", max_steps=3))
print("===== look at step by step by ==============")
# Inspect memory (audit trail)
for turn in agent.memory[-6:]:
    print(turn)


{'final_answer': "Sorry, I couldn't complete this with the available steps."}
{'final_answer': 'Your 2 scoops of vanilla ice cream have been purchased for a total of $240. Remaining stock: 19'}
===== look at step by step by ==============
{'role': 'user', 'content': 'I want to buy 2 scoops of vanilla ice cream.'}
{'role': 'selection', 'content': {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}}}
{'role': 'observation', 'content': {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}, 'result': {'status': 'purchased', 'item': {'name': 'vanilla', 'sugar_free': False, 'kcal': 207, 'price': 120, 'stock_quantity': 19}, 'total_cost': 240}}}
{'role': 'selection', 'content': {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}}}
{'role': 'observation', 'content': {'tool': 'purchase_icecream', 'args': {'name': 'vanilla', 'quantity': 2}, 'result': {'status': 'purchased', 'item': {'name': 'vanilla', 'sugar_free': False, 'kcal': 207, '

## Memory  for the Agent

#### Memory is lost after fixed number of turns; even if it is persisted to disk.

In [66]:
# last output 
# ReAct Notebook Chat. Type 'exit' to stop.

# Agent: We sell the following ice cream flavors: vanilla, strawberry, mint chocolate chip, pistachio (sugar-free), and cookies and cream.
# Agent: Vanilla ice cream: not sugar-free, 207 kcal, price 120, stock 21.
# Agent: Cookies and cream ice cream: not sugar-free, 240 kcal, price 150, stock 18.
# Goodbye!


# Agent: Hello Bhujay!
# Agent: Vanilla ice cream: 120
# Agent: You're welcome!
# Goodbye!

# You entered: tell me a one line story
# Agent: On a stormy night, the lighthouse keeper bravely guided a lost ship to safety.
# You entered: thats great , tell me one on ancientt wisdom
# Agent: In a time long past, a wise elder taught that true strength lies not in power, but in the courage to listen and learn.
# You entered: how old are you
# Agent: I don't have an age; I'm an AI created by OpenAI.
# You entered: when you were born
# Agent: I was launched by OpenAI in November 2022.
# You entered: do you have any siblings
# Agent: As an AI, I don't have siblings, but there are other AI models developed by OpenAI.
# You entered: who are they
# Agent: Other AI models developed by OpenAI include GPT-3, DALL-E, and Codex.
# You entered: what icecream do you cell
# Agent: We sell vanilla, strawberry, mint chocolate chip, pistachio (sugar-free), and cookies and cream ice cream.
# You entered: one line poen in kanada
# Agent: ಚಂದ್ರರ ಸುರ್ಯನಿಸು, ಪ್ರೀತಿಯ ಬೆಳಕು ಹರಿಯುತ್ತದೆ.
# You entered: one line in german
# Agent: Die Blumen blühen im Sonnenlicht.
# You entered: one in frfrench
# Agent: Les étoiles brillent dans la nuit.
# You entered: one in english
# Agent: The sun sets, painting the sky in hues of gold.
# You entered: what is my name
# ...
# You entered: wwhat ice cream dod you sell last time
# Agent: We sold vanilla, strawberry, mint chocolate chip, pistachio (sugar-free), and cookies and cream ice cream last time.
# You entered: exit
# Goodbye!

last output 
ReAct Notebook Chat. Type 'exit' to stop.

Agent: We sell the following ice cream flavors: vanilla, strawberry, mint chocolate chip, pistachio (sugar-free), and cookies and cream.
Agent: Vanilla ice cream: not sugar-free, 207 kcal, price 120, stock 21.
Agent: Cookies and cream ice cream: not sugar-free, 240 kcal, price 150, stock 18.
Goodbye!


Agent: Hello Bhujay!
Agent: Vanilla ice cream: 120
Agent: You're welcome!
Goodbye!

You entered: tell me a one line story
Agent: On a stormy night, the lighthouse keeper bravely guided a lost ship to safety.
You entered: thats great , tell me one on ancientt wisdom
Agent: In a time long past, a wise elder taught that true strength lies not in power, but in the courage to listen and learn.
You entered: how old are you
Agent: I don't have an age; I'm an AI created by OpenAI.
You entered: when you were born
Agent: I was launched by OpenAI in November 2022.
You entered: do you have any siblings
Agent: As an AI, I don't have siblings, but there are other AI models developed by OpenAI.
You entered: who are they
Agent: Other AI models developed by OpenAI include GPT-3, DALL-E, and Codex.
You entered: what icecream do you cell
Agent: We sell vanilla, strawberry, mint chocolate chip, pistachio (sugar-free), and cookies and cream ice cream.
You entered: one line poen in kanada
Agent: ಚಂದ್ರರ ಸುರ್ಯನಿಸು, ಪ್ರೀತಿಯ ಬೆಳಕು ಹರಿಯುತ್ತದೆ.
You entered: one line in german
Agent: Die Blumen blühen im Sonnenlicht.
You entered: one in frfrench
Agent: Les étoiles brillent dans la nuit.
You entered: one in english
Agent: The sun sets, painting the sky in hues of gold.
You entered: what is my name
...
You entered: wwhat ice cream dod you sell last time
Agent: We sold vanilla, strawberry, mint chocolate chip, pistachio (sugar-free), and cookies and cream ice cream last time.
You entered: exit
Goodbye!

### VECTOR STORE FOR RAG

#### set the embedding models 

In [67]:
import os
import json
import time
from typing import List, Dict, Any, Optional

import numpy as np


EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = openai_client


#### vector store

In [68]:
class SimpleVectorStore:
    """
    Minimal in-memory + on-disk vector store.
    Stores: [{"id", "text", "meta", "embedding"}].
    Uses OpenAI embeddings and persists to JSON on disk.
    """

    def __init__(self, client,  embed_model: str = EMBED_MODEL, path: str = "vec_store.json", verbose: bool = False):
        self.client = client
        self.embed_model = embed_model
        self.path = path
        self.verbose = verbose
        self.docs: List[Dict[str, Any]] = []
        self._load()

    # ---------- persistence ----------
    def _load(self) -> None:
        if self.path and os.path.exists(self.path):
            try:
                with open(self.path, "r", encoding="utf-8") as f:
                    raw = json.load(f)
                self.docs = []
                for d in raw:
                    emb = np.array(d["embedding"], dtype="float32")
                    self.docs.append({
                        "id": d["id"],
                        "text": d["text"],
                        "meta": d.get("meta", {}),
                        "embedding": emb,
                    })
                if self.verbose:
                    print(f"[VEC] Loaded {len(self.docs)} docs from {self.path}")
            except Exception as e:
                print(f"[VEC] Failed to load vec store: {e}")
                self.docs = []
        else:
            self.docs = []
            if self.verbose:
                print(f"[VEC] No vec store found at {self.path}, starting empty.")

    def _save(self) -> None:
        if not self.path:
            return
        serializable = []
        for d in self.docs:
            serializable.append({
                "id": d["id"],
                "text": d["text"],
                "meta": d.get("meta", {}),
                "embedding": d["embedding"].tolist(),
            })
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(serializable, f, ensure_ascii=False, indent=2)
        if self.verbose:
            print(f"[VEC] Saved {len(self.docs)} docs to {self.path}")

    # ---------- embeddings ----------
    def _embed(self, texts: List[str]) -> List[np.ndarray]:
        """
        Text -> dense vectors using OpenAI embeddings.
        """
        resp = self.client.embeddings.create(
            model=self.embed_model,
            input=texts,
        )
        return [np.array(d.embedding, dtype="float32") for d in resp.data]

    # ---------- public API ----------
    def add_documents(self, docs: List[Dict[str, Any]]) -> None:
        """
        Add docs with fields:
        - id   : string
        - text : string
        - meta : dict (optional)
        """
        if not docs:
            return
        texts = [d["text"] for d in docs]
        embs = self._embed(texts)

        for doc, emb in zip(docs, embs):
            self.docs.append({
                "id": doc.get("id"),
                "text": doc["text"],
                "meta": doc.get("meta", {}),
                "embedding": emb,
            })
        if self.verbose:
            print(f"[VEC] Added {len(docs)} docs. Total now: {len(self.docs)}")
        self._save()

    def search(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        """
        Cosine similarity search:
          1. Embed query.
          2. Normalize query + docs.
          3. Compute sim(q, v) = q·v after normalization.
          4. Return top-k docs.
        """
        if not self.docs:
            if self.verbose:
                print("[VEC] search: store empty.")
            return []

        q_emb = self._embed([query])[0]
        q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-10)

        sims = []
        for d in self.docs:
            v = d["embedding"]
            v = v / (np.linalg.norm(v) + 1e-10)
            sims.append(float(q_emb @ v))

        sims = np.array(sims)
        idxs = sims.argsort()[::-1][:k]

        results = []
        for i in idxs:
            d = self.docs[i]
            results.append({
                "id": d["id"],
                "text": d["text"],
                "meta": d["meta"],
                "score": float(sims[i]),
            })
        return results


#### JSON SHORT AND  LONG TERM MEMORY WITH RAG

In [69]:
class JSONMemory:
    """
    Short-term memory stored in a JSON file.

    - Holds up to `max_messages` recent messages.
    - When length > max_messages:
        -> Offload all but the last 2 *non-retrieved* messages
           into the vector store as small conversation docs.
    """

    def __init__(
        self,
        path: str = "agent_memory.json",
        max_messages: int = 6,
        long_term_store: Optional[SimpleVectorStore] = None,
        verbose: bool = False,
    ):
        self.path = path
        self.max_messages = max_messages
        self.long_term_store = long_term_store
        self.verbose = verbose
        self.messages: List[Dict[str, Any]] = []
        self.load()

    # ---------- persistence ----------
    def load(self) -> None:
        if os.path.exists(self.path):
            try:
                with open(self.path, "r", encoding="utf-8") as f:
                    self.messages = json.load(f)
                if self.verbose:
                    print(f"[MEM] Loaded {len(self.messages)} messages from {self.path}")
            except Exception as e:
                print(f"[MEM] Failed to load memory: {e}")
                self.messages = []
        else:
            self.messages = []
            if self.verbose:
                print(f"[MEM] No memory file at {self.path}, starting empty.")

    def save(self) -> None:
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.messages, f, ensure_ascii=False, indent=2)

    # ---------- public API ----------
    def add(self, role: str, content: str, meta: Optional[Dict[str, Any]] = None) -> None:
        """
        Add a message; if too many messages, offload older ones.
        """
        if meta is None:
            meta = {}
        self.messages.append({"role": role, "content": content, "ts": time.time(), "meta": meta})
        if self.verbose:
            print(f"[MEM] +{role}, len = {len(self.messages)}")

        if self.long_term_store and len(self.messages) > self.max_messages:
            self._offload_old_messages()

        self.save()

    def get_recent(self, k: int = 10) -> List[Dict[str, Any]]:
        return self.messages[-k:]

    # ---------- offload ----------
    def _offload_old_messages(self) -> None:
        """
        Offload older messages into vector store as small docs.

        Strategy:
        - Keep last 2 messages (whatever they are).
        - Consider all earlier messages as 'chunk'.
        - Filter out messages with meta.type == 'retrieved' (we don't offload those).
        - Build docs as user+assistant pairs.
        """
        if len(self.messages) <= 2 or not self.long_term_store:
            return

        total_before = len(self.messages)
        num_to_keep = 2
        num_to_offload = total_before - num_to_keep

        chunk = self.messages[:num_to_offload]
        remaining = self.messages[num_to_offload:]

        # Filter out retrieved snippets from offload
        base_chunk = [
            m for m in chunk
            if m.get("meta", {}).get("type") != "retrieved"
        ]

        if self.verbose:
            print(
                f"[MEM] Offloading {len(base_chunk)} msgs (skipping retrieved) "
                f"to vec store; keeping last {num_to_keep}."
            )

        docs: List[Dict[str, Any]] = []
        i = 0
        while i < len(base_chunk):
            m = base_chunk[i]
            if m["role"] == "user":
                user_text = m["content"]
                asst_text = None
                # Find next assistant after this user
                j = i + 1
                while j < len(base_chunk):
                    if base_chunk[j]["role"] == "assistant":
                        asst_text = base_chunk[j]["content"]
                        break
                    j += 1

                if asst_text:
                    text = f"User: {user_text}\nAssistant: {asst_text}"
                    next_i = j + 1
                else:
                    text = f"User: {user_text}"
                    next_i = i + 1

                doc_id = f"convpair_{int(time.time())}_{i}"
                docs.append({
                    "id": doc_id,
                    "text": text,
                    "meta": {
                        "source": "conversation",
                        "has_assistant": bool(asst_text),
                        "start_index": i,
                    },
                })
                i = next_i
            else:
                # skip non-user messages when forming pairs
                i += 1

        if docs:
            self.long_term_store.add_documents(docs)

        self.messages = remaining
        if self.verbose:
            print(f"[MEM] Offload complete. New len = {len(self.messages)}")
        self.save()

    # ---------- prepend retrieved snippet ----------
    def prepend_retrieved_snippet(self, text: str, hit: Dict[str, Any]) -> None:
        """
        Insert retrieved snippet at beginning of memory.
        Mark it as meta.type = 'retrieved' so it is never offloaded again.
        """
        msg = {
            "role": "assistant",
            "content": f"[Retrieved memory]\n{text}",
            "ts": time.time(),
            "meta": {"type": "retrieved", "source_id": hit.get("id")},
        }
        self.messages.insert(0, msg)
        self.save()


#### Utility functions – clear & inspect

In [70]:
def clear_memory_files(mem_path: str = "agent_memory.json", vec_path: str = "vec_store.json"):
    """
    Delete short-term and long-term memory files, if they exist.
    """
    for p, label in [(mem_path, "MEM"), (vec_path, "VEC")]:
        if os.path.exists(p):
            os.remove(p)
            print(f"[{label}] Deleted {p}")
        else:
            print(f"[{label}] No file at {p} to delete.")


def show_short_term_memory(mem: JSONMemory):
    print("=== Short-term JSON Memory ===")
    for i, m in enumerate(mem.messages):
        role = m.get("role", "user")
        text = m.get("content", "").replace("\n", " / ")
        meta = m.get("meta", {})
        print(f"{i:02d}. {role}: {text}  (meta={meta})")


def show_vec_store(vec: SimpleVectorStore, max_chars: int = 80):
    print("=== Vector Store Docs ===")
    for i, d in enumerate(vec.docs):
        first_line = d["text"].split("\n")[0][:max_chars]
        print(f"{i:02d}. id={d['id']} meta={d.get('meta',{})} text='{first_line}...'")


#### Initialize short and long term memory 

In [71]:
# To start from scratch:
clear_memory_files()

vec_store = SimpleVectorStore(client=openai_client, embed_model=EMBED_MODEL, path="vec_store.json", verbose=True)
mem = JSONMemory(path="agent_memory.json", max_messages=6, long_term_store=vec_store, verbose=True)

show_short_term_memory(mem)
show_vec_store(vec_store)


[MEM] No file at agent_memory.json to delete.
[VEC] Deleted vec_store.json
[VEC] No vec store found at vec_store.json, starting empty.
[MEM] No memory file at agent_memory.json, starting empty.
=== Short-term JSON Memory ===
=== Vector Store Docs ===


#### seed some text to test store and retrieval

In [72]:
KB_DOCS = [
    {
        "id": "kb1",
        "text": "My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.",
        "meta": {"topic": "my profile"}
    },
    {
        "id": "kb2",
        "text": "I love Alliance University because of its diverse community, excellent faculty, and strong emphasis on research and innovation.",
        "meta": {"topic": "university"}
    },
]

vec_store.add_documents(KB_DOCS)

def search_rag_conversations(query: str, top_k: int = 1):
    """
    Search both:
      - seed KB docs, and
      - offloaded conversation snippets (user+assistant pairs)
    in vec_store using embeddings.

    Returns a list of hit dicts with id, text, meta, score.
    """
    hits = vec_store.search(query, k=top_k)
    return [
        {
            "id": h["id"],
            "text": h["text"],
            "meta": h.get("meta", {}),
            "score": h["score"],
        }
        for h in hits
    ]

print(search_rag_conversations("Tell me about Bhujay's interests."))
print(search_rag_conversations("which university is good in India."))

[VEC] Added 2 docs. Total now: 2
[VEC] Saved 2 docs to vec_store.json
[{'id': 'kb1', 'text': 'My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.', 'meta': {'topic': 'my profile'}, 'score': 0.5599715709686279}]
[{'id': 'kb2', 'text': 'I love Alliance University because of its diverse community, excellent faculty, and strong emphasis on research and innovation.', 'meta': {'topic': 'university'}, 'score': 0.36538538336753845}]


#### AGENT WITH BOTH SHORT TERM AND LONG TERM MEMORY

In [82]:
import json
from typing import Dict, Any, List, Optional, Callable

class ReActAgent2:
    def __init__(
        self,
        client,           
        tool_map: Dict[str, Callable],
        tools_block: str,
        model: str = model,     
        embed_model: str = EMBED_MODEL,
        memory: Optional[JSONMemory] = None,
        history_messages: int = 6,
        system_preamble: str = (
            "You are a helpful assistant.\n"
            "- You have SHORT-TERM memory (recent messages).\n"
            "- You also receive RETRIEVED MEMORY blocks.\n"
            "- If RETRIEVED MEMORY contains facts about the user "
            " (e.g., their name, preferences), you MUST trust and use it."
        ),
        long_term_store: Optional[SimpleVectorStore] = None,
    ):
        self.client = client
        self.embed_model = embed_model
        self.model = model
        self.tool_map = tool_map
        self.tools_block = tools_block
        self.tool_names = ", ".join(tool_map.keys())

        self.memory = memory
        self.history_messages = history_messages
        self.system_preamble = system_preamble

        self.long_term_store = long_term_store

        self.selection_system = self._make_selection_system()
        self.final_system = (
            "You are finalizing the answer for the user.\n"
            "You may see a RETRIEVAL_CONTEXT with facts from memory.\n"
            "If RETRIEVAL_CONTEXT contains the user's name or profile, "
            "YOU MUST USE IT rather than saying you don't know.\n"
            "Return ONLY this JSON (no extra text): "
            '{"final_answer": "<concise helpful answer>"}'
        )

        # will store the last retrieved snippet for this turn
        self._last_retrieved_text: Optional[str] = None

    # ---------- public API ----------
    def chat(self, user_query: str, max_steps: int = 3) -> Dict[str, Any]:
        # 1) Save user message (may trigger offload)
        if self.memory:
            self.memory.add("user", user_query)

        # Reset last retrieval for this turn
        self._last_retrieved_text = None

        # 2) Retrieve exactly one best snippet from vec store
        if self.long_term_store:
            hits = self.long_term_store.search(user_query, k=1)
            if hits:
                hit = hits[0]
                self._last_retrieved_text = hit["text"]  # store for later

                print(f"[RAG] Retrieved doc id={hit['id']} score={hit['score']:.3f}")
                print("[RAG] Retrieved snippet:")
                print(hit["text"])
                print("-" * 40)

                # Also push into short-term memory, marked as retrieved
                if self.memory:
                    self.memory.prepend_retrieved_snippet(hit["text"], hit)

        # 3) Build context from short-term memory (including retrieved msgs)
        context_msgs = self._context_messages()

        # 4) ReAct: select tool -> run tool -> finalize
        selection = self._select_tool(user_query, context_msgs)
        tool = selection.get("tool", "")
        args = selection.get("args", {})
        observation = self._run_tool(tool, args)

        final = self._finalize(user_query, observation, context_msgs)
        if "final_answer" not in final:
            final = {"final_answer": "Sorry, I could not complete that with the tools."}

        # 5) Save assistant reply to memory
        if self.memory:
            self.memory.add("assistant", final["final_answer"])

        return final

    # ---------- internals ----------
    def _make_selection_system(self) -> str:
        return f"""
You are a ReAct-style assistant. You must choose ONE tool and JSON args.

TOOLS:
{self.tools_block}

Return STRICT JSON (no extra text, no code fences) with schema:
{{
  "tool": "<one of: {self.tool_names}>",
  "args": <object with only valid keys for that tool>
}}
"""

    def _context_messages(self) -> List[Dict[str, str]]:
        """
        Build chat history to send to the model for tool selection.

        We make sure that:
        - All RETRIEVED messages (meta.type == 'retrieved') are included.
        - Plus the last `history_messages` non-retrieved messages.
        """
        msgs: List[Dict[str, str]] = [
            {"role": "system", "content": self.system_preamble}
        ]

        if not self.memory:
            return msgs

        # Separate retrieved vs normal messages
        retrieved_msgs = []
        normal_msgs = []
        for m in self.memory.messages:
            meta = m.get("meta", {})
            if meta.get("type") == "retrieved":
                retrieved_msgs.append(m)
            else:
                normal_msgs.append(m)

        # Add all retrieved snippets first (they are important context)
        for m in retrieved_msgs:
            role = m.get("role", "assistant")
            content = m.get("content", "")
            if role in ("system", "user", "assistant"):
                msgs.append({"role": role, "content": content})

        # Then add recent normal messages (tail)
        tail = normal_msgs[-self.history_messages:]
        for m in tail:
            role = m.get("role", "user")
            content = m.get("content", "")
            if role in ("system", "user", "assistant"):
                msgs.append({"role": role, "content": content})

        return msgs

    def _select_tool(self, query: str, context_msgs: List[Dict[str, str]]) -> Dict[str, Any]:
        messages = context_msgs + [
            {"role": "system", "content": self.selection_system},
            {"role": "user", "content": query},
        ]
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            response_format={"type": "json_object"},
        )
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {"tool": "", "args": {}}

    def _run_tool(self, tool: str, args: Dict[str, Any]) -> Any:
        fn = self.tool_map.get(tool)
        if not fn:
            return {"error": f"Unknown tool '{tool}'"}
        try:
            return fn(**args)
        except TypeError as e:
            return {"error": f"Bad args for {tool}: {e}"}
        except Exception as e:
            return {"error": f"{type(e).__name__}: {e}"}

    def _finalize(self, query: str, observation: Any, context_msgs: List[Dict[str, str]]) -> Dict[str, Any]:
        """
        Final answer step: we explicitly pass the retrieved snippet (if any)
        as RETRIEVAL_CONTEXT so the model is forced to see it.
        """
        retrieval_block = ""
        if self._last_retrieved_text:
            retrieval_block = (
                "RETRIEVAL_CONTEXT (facts from memory):\n"
                f"{self._last_retrieved_text}\n\n"
            )

        user_content = (
            f"{retrieval_block}"
            f"User request: {query}\n"
            f"Observation (tool result):\n"
            f"{json.dumps(observation, ensure_ascii=False)}"
        )

        messages = context_msgs + [
            {"role": "system", "content": self.final_system},
            {"role": "user", "content": user_content},
        ]

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            response_format={"type": "json_object"},
        )
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {}


#### initialize the Agent

In [83]:
vec_store = SimpleVectorStore(client=openai_client, embed_model=EMBED_MODEL, path="vec_store.json", verbose=True)
mem = JSONMemory(path="agent_memory.json", max_messages=6, long_term_store=vec_store, verbose=True)

agent2 = ReActAgent2(
    client=openai_client,    
    tool_map=TOOL_MAP,
    tools_block=TOOLS_BLOCK,
    model=EMBED_MODEL,
    memory=mem,
    history_messages=6,
    system_preamble="You are a helpful, precise assistant.",
    long_term_store=vec_store,
)

# 4) Inspect initial state (only if you want to show this in class)
show_short_term_memory(mem)
show_vec_store(vec_store)

[VEC] Loaded 2 docs from vec_store.json
[MEM] Loaded 2 messages from agent_memory.json
=== Short-term JSON Memory ===
00. assistant: [Retrieved memory] / My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.  (meta={'type': 'retrieved', 'source_id': 'kb1'})
01. user: hi  (meta={})
=== Vector Store Docs ===
00. id=kb1 meta={'topic': 'my profile'} text='My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI ...'
01. id=kb2 meta={'topic': 'university'} text='I love Alliance University because of its diverse community, excellent faculty, ...'


In [ ]:
agent2.client.embeddings

### Run the Agent with Long term RAG Memory

In [84]:
import builtins
builtins.input = input
# --- Minimal Jupyter Chat Loop ---
print("Once the while loop starts , just keep typing in your input without waiting for a input cell\n")

print("ReAct Notebook Chat. Type 'bye', 'exit' or 'quit' to stop.\n")

while True:
    try:
        user_text = input("You: ").strip()
        print("You entered:", user_text)
    except EOFError:
        break

    if user_text.lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    result = agent2.chat(user_text, max_steps=3)
    answer = result.get("final_answer", "(no answer)")
        
    print("Agent:", answer)

Once the while loop starts , just keep typing in your input without waiting for a input cell

ReAct Notebook Chat. Type 'bye', 'exit' or 'quit' to stop.

You entered: hi
[MEM] +user, len = 3
[RAG] Retrieved doc id=kb1 score=0.266
[RAG] Retrieved snippet:
My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.
----------------------------------------


PermissionDeniedError: Error code: 403 - {'error': {'message': 'You are not allowed to sample from this model', 'type': 'invalid_request_error', 'param': None, 'code': None}}

#### Agent worked with log term memmory 

In [ ]:
# Once the while loop starts , just keep typing in your input without waiting for a input cell

# ReAct Notebook Chat. Type 'bye', 'exit' or 'quit' to stop.

# You entered: what did you sell me last time
# [MEM] +user, len = 2
# [RAG] Retrieved doc id=kb1 score=0.181
# [RAG] Retrieved snippet:
# My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 4
# Agent: Last time, you purchased pistachio ice cream, which is sugar-free and contains 180 kcal.
# You entered: apart from ice cream what i like
# [MEM] +user, len = 5
# [RAG] Retrieved doc id=kb1 score=0.477
# [RAG] Retrieved snippet:
# My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 7
# [MEM] Offloading 3 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 1 docs. Total now: 3
# [VEC] Saved 3 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: Apart from ice cream, you like learning Linear Algebra, building AI applications, classical guitars, and playing badminton.
# You entered: my name?
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=kb1 score=0.314
# [RAG] Retrieved snippet:
# My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: Your name is Bhujay.
# You entered: my friends name is Subrata ### line 186 !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=kb1 score=0.329
# [RAG] Retrieved snippet:
# My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 5
# [VEC] Saved 5 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: Your friend's name is Subrata.
# You entered: give me a chocolate type incecream
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763274661_0 score=0.397
# [RAG] Retrieved snippet:
# User: 
# Assistant: Last time, you purchased pistachio ice cream, which is sugar-free and contains 180 kcal.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: You can try mint chocolate chip ice cream, which is chocolate-flavored and has 215 kcal.
# You entered: why are you telling me kcal ?
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763274661_0 score=0.384
# [RAG] Retrieved snippet:
# User: 
# Assistant: Last time, you purchased pistachio ice cream, which is sugar-free and contains 180 kcal.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 7
# [VEC] Saved 7 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: I mentioned kcal to provide you with information about the calorie content of the ice cream, which can help you make informed choices based on your dietary preferences.
# You entered: tell me a one line advise
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763274689_0 score=0.245
# [RAG] Retrieved snippet:
# User: apart from ice cream what i like
# Assistant: Apart from ice cream, you like learning Linear Algebra, building AI applications, classical guitars, and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: Pursue your passions with dedication, balancing learning and leisure for a fulfilling life.
# You entered: on eline story
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763274689_0 score=0.131
# [RAG] Retrieved snippet:
# User: apart from ice cream what i like
# Assistant: Apart from ice cream, you like learning Linear Algebra, building AI applications, classical guitars, and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 9
# [VEC] Saved 9 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: In a quiet village, a young girl discovered a magical guitar that brought her dreams to life, transforming her world with melodies.
# You entered: I also have a suzuki mouth organ ### 225 @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=kb1 score=0.259
# [RAG] Retrieved snippet:
# My name is Bhujay, I love: ice cream,  learning Linear Algebra, and building AI applications, classical guitars and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: That's great, Bhujay! The Suzuki mouth organ can be a fun addition to your music collection.
# You entered: but why suzum=ki suddenly produced mouthoirgan , NI have heard their car bot in musical domain Yamaha  is more heard of
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763274689_0 score=0.222
# [RAG] Retrieved snippet:
# User: apart from ice cream what i like
# Assistant: Apart from ice cream, you like learning Linear Algebra, building AI applications, classical guitars, and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 11
# [VEC] Saved 11 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: Suzuki is known for various musical instruments, including the mouth organ, while Yamaha is more renowned for its pianos and electronic instruments. Both brands have made significant contributions to the musical world, but they specialize in different areas.
# You entered: what all do you remeber , can you summararize
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763274689_0 score=0.267
# [RAG] Retrieved snippet:
# User: apart from ice cream what i like
# Assistant: Apart from ice cream, you like learning Linear Algebra, building AI applications, classical guitars, and playing badminton.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: You enjoy ice cream, learning Linear Algebra, building AI applications, classical guitars, and playing badminton. You also have an interest in classic literature, with notable works including 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams, 'Foundation' and 'I, Robot' by Isaac Asimov.
# You entered: give your memory , each point as one new line
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763274798_2 score=0.317
# [RAG] Retrieved snippet:
# User: tell me a one line advise
# Assistant: Pursue your passions with dedication, balancing learning and leisure for a fulfilling life.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 13
# [VEC] Saved 13 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: You enjoy ice cream.
# You like learning Linear Algebra.
# You build AI applications.
# You play classical guitar.
# You enjoy playing badminton.
# You appreciate classic literature, including 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams.
# You also like 'Foundation' and 'I, Robot' by Isaac Asimov.
# You entered: apart from me what do you remember about the converastion
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763274957_2 score=0.409
# [RAG] Retrieved snippet:
# User: what all do you remeber , can you summararize
# Assistant: You enjoy ice cream, learning Linear Algebra, building AI applications, classical guitars, and playing badminton. You also have an interest in classic literature, with notable works including 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams, 'Foundation' and 'I, Robot' by Isaac Asimov.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: The conversation included your interests in ice cream, Linear Algebra, AI applications, classical guitars, badminton, and classic literature, specifically mentioning 'Foundation' (1951) and 'I, Robot' (1950) by Isaac Asimov.
# You entered: show each  point as a new line
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763274798_2 score=0.223
# [RAG] Retrieved snippet:
# User: tell me a one line advise
# Assistant: Pursue your passions with dedication, balancing learning and leisure for a fulfilling life.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 15
# [VEC] Saved 15 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: You enjoy ice cream.
# You like learning Linear Algebra.
# You build AI applications.
# You play classical guitar.
# You enjoy playing badminton.
# You appreciate classic literature, including 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams.
# You also like 'Foundation' by Isaac Asimov (1951).
# You also like 'I, Robot' by Isaac Asimov (1950).
# You entered: give me a the first book
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763275006_0 score=0.274
# [RAG] Retrieved snippet:
# User: give your memory , each point as one new line
# Assistant: You enjoy ice cream.
# You like learning Linear Algebra.
# You build AI applications.
# You play classical guitar.
# You enjoy playing badminton.
# You appreciate classic literature, including 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams.
# You also like 'Foundation' and 'I, Robot' by Isaac Asimov.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: The first book is 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams.
# You entered: my friends name
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763274736_0 score=0.548
# [RAG] Retrieved snippet:
# User: my friends name is Subrata !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# Assistant: Your friend's name is Subrata.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 17
# [VEC] Saved 17 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: Your friend's name is Subrata.
# You entered: give me the last icecream that I took
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763274661_0 score=0.519
# [RAG] Retrieved snippet:
# User: 
# Assistant: Last time, you purchased pistachio ice cream, which is sugar-free and contains 180 kcal.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: The last ice cream you took was pistachio ice cream, which is sugar-free and contains 180 kcal.
# You entered: no you suggested something else
# [MEM] +user, len = 6
# [RAG] Retrieved doc id=convpair_1763275006_2 score=0.266
# [RAG] Retrieved snippet:
# User: apart from me what do you remember about the converastion
# Assistant: The conversation included your interests in ice cream, Linear Algebra, AI applications, classical guitars, badminton, and classic literature, specifically mentioning 'Foundation' (1951) and 'I, Robot' (1950) by Isaac Asimov.
# ----------------------------------------
# [MEM] +assistant, len = 8
# [MEM] Offloading 4 msgs (skipping retrieved) to vec store; keeping last 2.
# [VEC] Added 2 docs. Total now: 19
# [VEC] Saved 19 docs to vec_store.json
# [MEM] Offload complete. New len = 2
# Agent: You previously mentioned pistachio ice cream, but I also suggested vanilla ice cream as another option.
# You entered: what mouth organ I have
# [MEM] +user, len = 3
# [RAG] Retrieved doc id=convpair_1763274886_2 score=0.522
# [RAG] Retrieved snippet:
# User: I also have a suzuki mouth organ
# Assistant: That's great, Bhujay! The Suzuki mouth organ can be a fun addition to your music collection.
# ----------------------------------------
# [MEM] +assistant, len = 5
# Agent: You have a Suzuki mouth organ. ##89 @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
# You entered: bye
# Goodbye!